In [10]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
from tqdm import tqdm
import time

In [11]:
import torchvision.models as models

class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Pretrained ResNet18 backbone
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        
        # Remove final classification layer
        self.backbone.fc = nn.Identity()
        
        # Regression head (8 values = 4 corners)
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 8)  # 4 corners (x,y)
        )

    def forward(self, x):
        features = self.backbone(x)
        corners = self.head(features)
        return corners, None  # keep your interface consistent

In [13]:
class ShadowDataset(Dataset):
    def __init__(self, folder, img_size=224):
        self.folder = folder
        self.img_size = img_size
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],  # ImageNet mean
                        [0.229, 0.224, 0.225])   # ImageNet std
        ])
        
        # Collect all images that have a matching json
        self.samples = [
            f.replace(".png", "")
            for f in os.listdir(folder)
            if f.endswith(".png") and 
               os.path.exists(os.path.join(folder, f.replace(".png", ".json")))
        ]
        print(f"Found {len(self.samples)} samples in {folder}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        
        # ── Load image ──
        img_path = os.path.join(self.folder, f"{name}.png")
        img = Image.open(img_path).convert("RGB")
        W, H = img.size          # original size, needed for normalizing
        img = self.transform(img)
        
        # ── Load annotation ──
        json_path = os.path.join(self.folder, f"{name}.json")
        with open(json_path) as f:
            ann = json.load(f)
        
        # Normalize all corner coordinates to 0-1
        # Note: values CAN be > 1.0 since person is off-screen!
        bbox = ann["bbox"]
        corners = torch.tensor([
            bbox["top_left"][0]     / W,
            bbox["top_left"][1]     / H,
            bbox["top_right"][0]    / W,
            bbox["top_right"][1]    / H,
            bbox["bottom_left"][0]  / W,
            bbox["bottom_left"][1]  / H,
            bbox["bottom_right"][0] / W,
            bbox["bottom_right"][1] / H,
        ], dtype=torch.float32)

        direction = torch.tensor(
            [ann["walking_into_frame_bool"]], 
        dtype=torch.float32
)
        
        return img, corners, direction, W, H, name


In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# Load dataset
dataset = ShadowDataset("data/train_data/train_data")

# Split
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)

# Model
model = ShadowDetector().to(device)

# Loss + optimizer
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_val_loss = float("inf")

for epoch in range(50):
    start = time.time()

    # ── TRAIN ──
    model.train()
    train_losses = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/50 [Train]", leave=False)
    for imgs, corners, _, _, _, _ in loop:
        imgs    = imgs.to(device)
        corners = corners.to(device)

        optimizer.zero_grad()

        pred_corners, _ = model(imgs)   # ignore direction output

        loss = criterion(pred_corners, corners)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    # ── VALIDATE ──
    model.eval()
    val_losses = []

    loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/50 [Val]", leave=False)
    with torch.no_grad():
        for imgs, corners, _, _, _, _ in loop:
            imgs    = imgs.to(device)
            corners = corners.to(device)

            pred_corners, _ = model(imgs)

            loss = criterion(pred_corners, corners)

            val_losses.append(loss.item())
            loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_train = sum(train_losses) / len(train_losses)
    avg_val   = sum(val_losses) / len(val_losses)
    elapsed   = time.time() - start

    print(f"Epoch {epoch+1:02d}/50 | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Time: {elapsed:.1f}s")

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print("↑ Saved best model")

    scheduler.step()

Using: cuda
Found 1692 samples in data/train_data/train_data


KeyboardInterrupt: 